# Chapter 6 Tutorial — Limiting Behavior and Applications of Markov Chains

This notebook is a step-by-step tutorial for Chapter 6 of Çınlar, covering the material in the uploaded chapter excerpt:

1. computing the potential matrix \(R\) and the hitting-probability matrix \(F\);
2. limiting probabilities for recurrent classes;
3. invariant distributions and long-run averages;
4. periodic states and cyclic decomposition.

The notebook includes the chapter's examples, proof ideas, and executable computations.

In [ ]:
import numpy as np
import pandas as pd
from fractions import Fraction
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

def mat_power(P, n):
    return np.linalg.matrix_power(np.asarray(P, dtype=float), n)

def stationary_distribution(P):
    # Solve pi = pi P, sum pi = 1.
    # Works best for one irreducible finite recurrent class.
    P = np.asarray(P, dtype=float)
    n = P.shape[0]
    A = (P.T - np.eye(n))
    A[-1, :] = 1.0
    b = np.zeros(n)
    b[-1] = 1.0
    return np.linalg.solve(A, b)

def potential_matrix(Q):
    # S = I + Q + Q^2 + ... = (I-Q)^(-1), for transient Q.
    Q = np.asarray(Q, dtype=float)
    return np.linalg.inv(np.eye(Q.shape[0]) - Q)

def fmt_frac_matrix(A, max_den=1000):
    return pd.DataFrame([[str(Fraction(float(x)).limit_denominator(max_den)) for x in row] for row in A])

def simulate_chain(P, x0, n, rng=None):
    rng = np.random.default_rng(rng)
    P = np.asarray(P, dtype=float)
    xs = [x0]
    for _ in range(n):
        xs.append(rng.choice(P.shape[0], p=P[xs[-1]]))
    return np.array(xs)

## 1. The two central objects: \(R\) and \(F\)

Let \(X = (X_n)\) be a Markov chain on a countable state space \(E\).

For states \(i,j \in E\):

\[
N_j = \sum_{n=0}^{\infty} 1_{\{X_n=j\}}
\]

is the total number of visits to \(j\).

The **potential matrix** is

\[
R(i,j) = E_i[N_j].
\]

The **hitting probability matrix** is

\[
F(i,j) = P_i(T_j < \infty),
\]

where

\[
T_j = \inf\{n \ge 1 : X_n = j\}
\]

is the first positive hitting time of \(j\).

Thinking model:

- \(F(i,j)\): “Will I ever reach \(j\)?”
- \(R(i,j)\): “How many total visits to \(j\) should I expect?”

### Recurrent and transient states

A state \(j\) is:

- **recurrent** if \(P_j(T_j < \infty)=1\);
- **transient** if \(P_j(T_j < \infty)<1\).

A recurrent state is visited infinitely often once it is reached. A transient state is visited only finitely many times almost surely.

Important consequences:

\[
R(j,j)=\infty \quad \text{if } j \text{ is recurrent},
\]

and

\[
R(j,j)<\infty \quad \text{if } j \text{ is transient}.
\]

## 1.1 Computing \(R\) for transient states

Suppose the transient states are collected in a set \(D\). Put all recurrent states first, and the transient states last. Then the transition matrix can be block-written as

\[
P =
\begin{pmatrix}
K & 0 \\
L & Q
\end{pmatrix}.
\]

Here \(Q\) is the transition matrix restricted to transient states.

For transient-to-transient visits,

\[
S = I + Q + Q^2 + \cdots.
\]

This is the transient part of the potential matrix \(R\):

\[
S(i,j) = R(i,j), \qquad i,j \in D.
\]

If there are finitely many transient states, then

\[
S = (I-Q)^{-1}.
\]

### Proof idea

Since

\[
S = I + Q + Q^2 + \cdots,
\]

multiplying by \(I-Q\) gives telescoping cancellation:

\[
(I-Q)S = I.
\]

So

\[
S=(I-Q)^{-1}.
\]

In [ ]:
# Example: a small transient submatrix Q.
Q = np.array([
    [0.2, 0.3],
    [0.1, 0.2],
])

S = potential_matrix(Q)
S

Interpretation: if the chain starts in transient state \(i\), then \(S(i,j)\) is the expected number of visits to transient state \(j\) before absorption into some recurrent class.

In [ ]:
fmt_frac_matrix(S)

## 1.2 Minimal nonnegative solution

The chapter also states that \(S\) is the minimal nonnegative solution of

\[
Y = I + QY.
\]

Equivalently,

\[
(I-Q)Y=I.
\]

### Proof idea

Starting from

\[
Y = I + QY,
\]

substitute the right side repeatedly:

\[
Y = I + Q(I+QY)=I+Q+Q^2Y,
\]

then

\[
Y = I+Q+Q^2+Q^3Y,
\]

and so on. After \(n\) steps,

\[
Y = I+Q+\cdots+Q^n + Q^{n+1}Y.
\]

If \(Y \ge 0\), then

\[
Y \ge I+Q+\cdots+Q^n.
\]

Letting \(n\to\infty\),

\[
Y \ge S.
\]

So \(S\) is the smallest nonnegative solution.

## 1.3 Example 1.12: computing the potential matrix

The chapter gives an eight-state chain whose states split into:

- recurrent class \(C_1=\{1,2,3\}\), recurrent and periodic;
- recurrent class \(C_2=\{4,5\}\), recurrent non-null and aperiodic;
- transient states \(D=\{6,7,8\}\).

The transient submatrix is

\[
Q =
\begin{pmatrix}
0.4 & 0.6 & 0\\
0 & 0 & 0.2\\
0.6 & 0 & 0
\end{pmatrix}.
\]

The transient potential block is

\[
S=(I-Q)^{-1}.
\]

In [ ]:
Q = np.array([
    [0.4, 0.6, 0.0],
    [0.0, 0.0, 0.2],
    [0.6, 0.0, 0.0],
])
S = potential_matrix(Q)
S, fmt_frac_matrix(S)

Thus, for example, starting from transient state \(6\), the expected number of visits to states \(6,7,8\) is given by the first row of \(S\).

The full potential matrix \(R\) has:

- \(\infty\) inside recurrent closed classes;
- \(0\) from recurrent classes to states outside their closed class;
- \(S\) in the transient-to-transient block;
- finite expected visits from transient states to recurrent states, depending on which recurrent class is reached.

## 1.4 Computing \(F\): probabilities of ever reaching states

For hitting probabilities, there are several cases:

\[
F(j,j)=1 \quad \text{if } j \text{ is recurrent},
\]

\[
F(j,j)<1 \quad \text{if } j \text{ is transient}.
\]

If \(i\) is recurrent and \(j\) is transient, then

\[
F(i,j)=0.
\]

If \(i,j\) are recurrent but belong to different closed recurrent classes, then

\[
F(i,j)=0.
\]

If \(i,j\) are transient, then the chapter gives

\[
F(j,j)=1-\frac{1}{R(j,j)},
\]

and for \(i\ne j\),

\[
F(i,j)=\frac{R(i,j)}{R(j,j)}.
\]

### Why \(F(i,j)=R(i,j)/R(j,j)\)

If starting from \(i\), the chain ever reaches \(j\), then from that point onward the expected number of visits to \(j\) is \(R(j,j)\). Therefore

\[
R(i,j)=F(i,j)R(j,j),
\]

so

\[
F(i,j)=\frac{R(i,j)}{R(j,j)}.
\]

In [ ]:
F_transient = np.zeros_like(S)
for i in range(S.shape[0]):
    for j in range(S.shape[1]):
        if i == j:
            F_transient[i, j] = 1 - 1 / S[j, j]
        else:
            F_transient[i, j] = S[i, j] / S[j, j]

F_transient, fmt_frac_matrix(F_transient)

## 1.5 Reaching recurrent classes from transient states

Suppose the recurrent classes are lumped into absorbing states. In block form,

\[
\bar P =
\begin{pmatrix}
I & 0 \\
B & Q
\end{pmatrix},
\]

where:

- \(Q\) is transient-to-transient;
- \(B\) is transient-to-recurrent-class.

Then the matrix of probabilities of eventually entering each recurrent class is

\[
G = SB,
\]

where

\[
S=(I-Q)^{-1}.
\]

### Proof idea

To enter a recurrent class \(C\), the chain may spend \(0,1,2,\ldots\) steps moving among transient states and then jump into \(C\). Therefore

\[
G = B + QB + Q^2B + \cdots = (I+Q+Q^2+\cdots)B = SB.
\]

In [ ]:
# Toy example: two absorbing classes A,B and three transient states.
Q = np.array([
    [0.2, 0.3, 0.0],
    [0.1, 0.2, 0.1],
    [0.0, 0.2, 0.2],
])
B = np.array([
    [0.4, 0.1],
    [0.3, 0.3],
    [0.1, 0.5],
])
print("row sums of [B Q] =", np.c_[B, Q].sum(axis=1))

S = potential_matrix(Q)
G = S @ B
G, G.sum(axis=1)

Each row of \(G\) sums to \(1\): starting from a transient state, the chain eventually reaches one of the recurrent classes.

## 1.6 Example 1.24 and 1.25: matrix of hitting probabilities

In the chapter examples, the recurrent classes are collapsed into absorbing states, then

\[
S=(I-Q)^{-1}, \qquad G=SB.
\]

The resulting \(F\) matrix is assembled as:

- \(1\) inside a recurrent class that can reach itself;
- \(0\) between different recurrent classes;
- \(G(i,C)\) for transient state \(i\) and recurrent class \(C\);
- transient-to-transient entries computed from \(S\).

Below is the reusable computation pattern.

In [ ]:
def absorbing_probabilities(Q, B):
    S = potential_matrix(Q)
    return S, S @ B

# Example 1.25-style reduced chain:
# Two recurrent classes collapsed to two absorbing states;
# two transient states.
Q = np.array([
    [0.7, 0.1],
    [0.2, 0.6],
])
B = np.array([
    [0.1, 0.1],
    [0.1, 0.1],
])
S, G = absorbing_probabilities(Q, B)
print("S =")
display(pd.DataFrame(S))
print("G =")
display(pd.DataFrame(G))
print("row sums G =", G.sum(axis=1))

## 1.7 Example 1.26: bankruptcy and eventual absorption

The chapter gives a business/bankruptcy interpretation.

Let \(X_n\) be the total assets of a firm after year \(n\). State \(0\) means bankruptcy. If state \(0\) is the only recurrent class and all positive states are transient, then

\[
P_i(\text{eventual bankruptcy}) = 1.
\]

But that does **not** mean the probability of bankruptcy after one year is \(1\). It means that, over an unlimited future horizon, bankruptcy occurs almost surely.

If

\[
g(i)=P_i(\text{ever reach }0),
\]

then in this setting

\[
g(i)=1,\qquad i=1,2,\ldots.
\]

The complementary quantity

\[
1-g(i)
\]

is the probability that the firm never goes bankrupt.

In [ ]:
# A finite gambler's-ruin style bankruptcy model.
# states 0..N, 0 is bankruptcy, N is success/absorbing cap.
N = 10
p_up = 0.45
p_down = 0.55

P = np.zeros((N+1, N+1))
P[0,0] = 1
P[N,N] = 1
for i in range(1, N):
    P[i, i+1] = p_up
    P[i, i-1] = p_down

# transient states 1..N-1
Q = P[1:N, 1:N]
B = P[1:N][:, [0, N]]
S = potential_matrix(Q)
G = S @ B

pd.DataFrame(G, index=range(1,N), columns=["bankruptcy", "upper absorbing state"])

This finite model has two absorbing classes, \(0\) and \(N\). The column named “bankruptcy” gives the probability of ever hitting \(0\) before hitting \(N\).

# 2. Recurrent states and limiting probabilities

For an irreducible, aperiodic Markov chain whose states are recurrent non-null, the limiting probabilities exist:

\[
\lim_{n\to\infty} P^n(i,j)=\pi(j).
\]

The vector \(\pi\) is the unique probability solution of

\[
\pi = \pi P,
\]

\[
\sum_j \pi(j)=1.
\]

This \(\pi\) is called the **invariant distribution** or **stationary distribution**.

Thinking model:

- \(\pi(j)\) is the long-run fraction of time spent in state \(j\).
- If the chain starts with distribution \(\pi\), then it stays distributed as \(\pi\) forever.

## 2.1 Theorem 2.1: limiting probabilities in an irreducible aperiodic recurrent chain

If \(X\) is irreducible and aperiodic, and all states are recurrent non-null, then \(\pi\) is determined by

\[
\pi(j)=\sum_i \pi(i)P(i,j), \qquad j\in E,
\]

\[
\sum_j \pi(j)=1.
\]

Then

\[
\pi(j)=\lim_{n\to\infty}P^n(i,j),
\]

for all \(i,j\).

### Proof idea

For a recurrent, non-null, aperiodic chain, the renewal structure of returns to a state forces the transition probabilities to settle into a stable long-run rhythm. The limiting vector must satisfy \(\pi=\pi P\), because

\[
P^{n+1}=P^nP,
\]

and taking limits gives

\[
\pi=\pi P.
\]

The normalization condition \(\sum_j \pi(j)=1\) makes \(\pi\) a probability distribution.

## 2.2 Example 2.15: solving \(\pi=\pi P\)

The chapter considers

\[
P =
\begin{pmatrix}
0.3 & 0.5 & 0.2\\
0.6 & 0 & 0.4\\
0 & 0.4 & 0.6
\end{pmatrix}.
\]

All states are recurrent, non-null, and aperiodic, so the limiting matrix has every row equal to \(\pi\).

In [ ]:
P = np.array([
    [0.3, 0.5, 0.2],
    [0.6, 0.0, 0.4],
    [0.0, 0.4, 0.6],
])

pi = stationary_distribution(P)
pi, fmt_frac_matrix([pi])

In [ ]:
for n in [1, 2, 5, 10, 25, 100]:
    print(f"P^{n}:")
    display(pd.DataFrame(mat_power(P, n)))

The rows of \(P^n\) converge to

\[
\pi=\left(\frac{6}{23},\frac{7}{23},\frac{10}{23}\right).
\]

## 2.3 Computational hint: one equation is redundant

The equations in

\[
\pi=\pi P
\]

are linearly dependent: once you have enough of them plus

\[
\sum_j\pi(j)=1,
\]

one of the original equations can be discarded.

Practical method:

1. write \(\pi=\pi P\);
2. replace one equation by \(\sum_j\pi(j)=1\);
3. solve the resulting linear system.

## 2.4 Example 2.16: multiple recurrent classes plus transient states

When a chain has several recurrent classes and transient states, \(P^n\) does not usually converge to a matrix whose rows are all identical.

Instead:

- inside each recurrent class, the chain converges to that class’s invariant distribution;
- transient states vanish in the limit;
- starting from a transient state, the limiting distribution is a weighted mixture of the recurrent-class invariant distributions, with weights equal to the probabilities of eventually entering each class.

Symbolically, for recurrent class \(C_k\),

\[
\lim_{n\to\infty}P^n(i,j)
=
F(i,C_k)\pi_k(j),
\qquad j\in C_k.
\]

In [ ]:
# Construct a small chain with:
# class C1 = {0,1}, class C2 = {2,3}, transient states {4,5}
P = np.array([
    [0.2, 0.8, 0,   0,   0,   0],
    [0.7, 0.3, 0,   0,   0,   0],
    [0,   0,   0.5, 0.5, 0,   0],
    [0,   0,   0.4, 0.6, 0,   0],
    [0.2, 0.0, 0.1, 0.0, 0.4, 0.3],
    [0.0, 0.1, 0.0, 0.2, 0.2, 0.5],
])

P100 = mat_power(P, 100)
pd.DataFrame(P100)

In [ ]:
pi1 = stationary_distribution(P[:2,:2])
pi2 = stationary_distribution(P[2:4,2:4])
pi1, pi2

In [ ]:
Q = P[4:6, 4:6]
B = P[4:6, :4]  # probabilities from transient states into recurrent states
S = potential_matrix(Q)
G_to_states = S @ B

# Sum by recurrent class
G_classes = np.c_[G_to_states[:, :2].sum(axis=1), G_to_states[:, 2:4].sum(axis=1)]
G_classes

Starting from a transient state, the limit row is:

\[
G(i,C_1)\pi_1 + G(i,C_2)\pi_2.
\]

In [ ]:
limit_from_transient = []
for row in G_classes:
    limit_from_transient.append(np.r_[row[0]*pi1, row[1]*pi2, 0, 0])
pd.DataFrame(limit_from_transient)

## 2.5 Example 2.17: reflected random walk

The chapter considers a Markov chain on \(\{0,1,\ldots\}\) with transition structure

\[
P =
\begin{pmatrix}
q & p & 0 & 0 & \cdots\\
q & 0 & p & 0 & \cdots\\
0 & q & 0 & p & \cdots\\
0 & 0 & q & 0 & p & \cdots\\
\vdots & \vdots & \vdots & \vdots & \ddots
\end{pmatrix},
\]

with \(p+q=1\).

The invariant equations give

\[
\pi_1 = \frac{p}{q}\pi_0,
\]

\[
\pi_2 = \left(\frac{p}{q}\right)^2\pi_0,
\]

and in general

\[
\pi_j = \left(\frac{p}{q}\right)^j\pi_0.
\]

This can be normalized only if

\[
p < q.
\]

Then

\[
\pi_j =
\left(1-\frac{p}{q}\right)
\left(\frac{p}{q}\right)^j,
\qquad j=0,1,2,\ldots.
\]

If \(p\ge q\), the chain is not recurrent non-null.

In [ ]:
p = 0.4
q = 0.6
rho = p/q
j = np.arange(20)
pi = (1-rho) * rho**j

plt.figure()
plt.stem(j, pi)
plt.xlabel("state j")
plt.ylabel(r"$\pi_j$")
plt.title("Invariant distribution for reflected random walk, p < q")
plt.show()

pi[:10], pi.sum()

## 2.6 Example 2.18: lifetime chain and age distribution

The chapter revisits a replacement/lifetime model.

Let \(p_k\) describe the lifetime distribution of a component. The invariant distribution can be written in terms of tail probabilities. If

\[
m = p_1 + 2p_2 + 3p_3 + \cdots
\]

is finite, then

\[
\pi(j)
=
\frac{1-p_1-\cdots-p_j}{m}.
\]

Interpretation:

- numerator: probability that a component survives beyond age \(j\);
- denominator: mean lifetime.

So long-run age distribution is proportional to the survival curve.

In [ ]:
# Example lifetime distribution on {1,2,3,...}: geometric with parameter r
r = 0.25
k = np.arange(1, 40)
p_k = r * (1-r)**(k-1)
m = np.sum(k * p_k)

j = np.arange(0, 20)
tail_after_j = np.array([np.sum(p_k[k > jj]) for jj in j])
pi_j = tail_after_j / m

plt.figure()
plt.stem(j, pi_j)
plt.xlabel("age j")
plt.ylabel(r"$\pi(j)$")
plt.title("Long-run age distribution is normalized survival")
plt.show()

pd.DataFrame({"j": j, "pi(j)": pi_j}).head(10)

# 2.7 Mean recurrence time

For a recurrent non-null aperiodic state \(j\), let

\[
m(j)=E_j[T_j]
\]

be the mean return time to \(j\). The chapter states

\[
\pi(j)=\frac{1}{m(j)}.
\]

Thinking model:

If the expected time between visits to \(j\) is \(m(j)\), then the long-run rate of visits to \(j\) is \(1/m(j)\).

In [ ]:
P = np.array([
    [0.3, 0.5, 0.2],
    [0.6, 0.0, 0.4],
    [0.0, 0.4, 0.6],
])
pi = stationary_distribution(P)
mean_return_times = 1 / pi
pd.DataFrame({
    "state": [0,1,2],
    "pi(j)": pi,
    "mean return time 1/pi(j)": mean_return_times
})

## 2.8 Long-run averages

If \(X\) is irreducible recurrent with limiting distribution \(\pi\), then for bounded \(f\),

\[
\lim_{n\to\infty}\frac{1}{n+1}\sum_{m=0}^{n}f(X_m)
=
\sum_j \pi(j)f(j)
\]

almost surely.

This is the Markov-chain version of the law of large numbers.

For expectations,

\[
\lim_{n\to\infty}\frac{1}{n+1}\sum_{m=0}^{n}E_i[f(X_m)]
=
\sum_j \pi(j)f(j).
\]

In [ ]:
P = np.array([
    [0.3, 0.5, 0.2],
    [0.6, 0.0, 0.4],
    [0.0, 0.4, 0.6],
])
pi = stationary_distribution(P)
f = np.array([10, 20, 100])
target = pi @ f

xs = simulate_chain(P, x0=0, n=20_000, rng=1)
running_avg = np.cumsum(f[xs]) / np.arange(1, len(xs)+1)

plt.figure()
plt.plot(running_avg)
plt.axhline(target, linestyle="--")
plt.xlabel("time n")
plt.ylabel("running average")
plt.title("Markov-chain long-run average")
plt.show()

target

## 2.9 Invariant measure for irreducible recurrent chains

For infinite chains, an invariant probability distribution may not exist. The chapter generalizes to an **invariant measure** \(v\), satisfying

\[
v = vP.
\]

If \(X\) is irreducible recurrent, a strictly positive invariant measure exists and is unique up to multiplication by a constant.

If the invariant measure has finite total mass, it can be normalized into an invariant probability distribution:

\[
\pi(j)=\frac{v(j)}{\sum_k v(k)}.
\]

If the total mass is infinite, the chain is recurrent but not positive recurrent.

# 3. Periodic states

The earlier convergence theorem required aperiodicity. Periodic chains behave differently.

A recurrent state \(j\) has period \(\delta\ge 2\) if returns to \(j\) are possible only at times that are multiples of \(\delta\).

For an irreducible chain, all states have the same period.

## 3.1 Cyclic decomposition

If an irreducible recurrent chain has period \(\delta\), the state space can be partitioned into cyclic classes

\[
B_0,B_1,\ldots,B_{\delta-1},
\]

such that every step moves from

\[
B_r \to B_{r+1 \pmod \delta}.
\]

Thus \(P^\delta\) leaves each \(B_r\) closed. The chain sampled every \(\delta\) steps becomes aperiodic within each cyclic class.

## 3.2 Example 3.3: a periodic irreducible chain

The chapter gives an irreducible chain with period \(3\). Its states split into three cyclic classes. After one step, the chain moves to the next class; after three steps, it returns to the same class.

The key computational point is:

\[
P^3 =
\begin{pmatrix}
P_1 & 0 & 0 \\
0 & P_2 & 0 \\
0 & 0 & P_3
\end{pmatrix},
\]

after states are ordered by cyclic class.

So \(P^3\) decomposes into separate aperiodic chains on the cyclic classes.

In [ ]:
# Simple 3-cycle with randomness inside classes.
# Classes: B0={0,1}, B1={2,3}, B2={4,5}
P = np.zeros((6,6))

# From B0 to B1
P[0, [2,3]] = [0.7, 0.3]
P[1, [2,3]] = [0.2, 0.8]

# From B1 to B2
P[2, [4,5]] = [0.5, 0.5]
P[3, [4,5]] = [0.9, 0.1]

# From B2 to B0
P[4, [0,1]] = [0.4, 0.6]
P[5, [0,1]] = [0.3, 0.7]

P3 = mat_power(P, 3)
pd.DataFrame(P3)

The zeros outside the diagonal blocks show that \(P^3\) keeps the chain inside its cyclic class.

## 3.3 Limiting behavior in the periodic case

For a periodic chain with period \(\delta\), the ordinary limit

\[
\lim_{n\to\infty}P^n(i,j)
\]

usually does not exist.

But limits along residue classes do exist. If \(i\in B_\alpha\) and \(j\in B_\beta\), then \(P^n(i,j)\) can be nonzero only when

\[
\beta \equiv \alpha+n \pmod \delta.
\]

For those valid subsequences,

\[
\lim_{m\to\infty}P^{m\delta+\ell}(i,j)
\]

exists and is determined by the invariant distribution of the corresponding closed class of \(P^\delta\).

This is why periodic chains oscillate rather than settle at every time step.

In [ ]:
# Show oscillation of P^n(0, j)
rows = []
for n in range(1, 16):
    rows.append(mat_power(P, n)[0])
pd.DataFrame(rows, index=range(1, 16))

In [ ]:
# But along multiples of 3, the row converges within B0.
for n in [3, 6, 9, 30, 90]:
    print(f"n = {n}")
    display(pd.DataFrame([mat_power(P, n)[0]]))

## 3.4 Theorem 3.7 and Theorem 3.10: applying aperiodic theory to \(P^\delta\)

Let \(P\) be irreducible recurrent with period \(\delta\), and let

\[
B_0,\ldots,B_{\delta-1}
\]

be its cyclic classes. Then \(P^\delta\) restricted to each \(B_r\) is irreducible and aperiodic.

Therefore the aperiodic limiting results from Section 2 apply to each block of \(P^\delta\).

The invariant distribution \(\pi\) still solves

\[
\pi=\pi P,
\qquad
\sum_j\pi(j)=1,
\]

but \(P^n(i,j)\) need not converge for all \(n\). Instead the chain has periodic subsequential limits.

In [ ]:
pi = stationary_distribution(P)
pi

In [ ]:
# Long-run time averages still converge to pi even though P^n rows oscillate.
xs = simulate_chain(P, x0=0, n=50_000, rng=2)
empirical = np.bincount(xs, minlength=6) / len(xs)
pd.DataFrame({"pi": pi, "empirical": empirical})

# Summary table

| Concept | Formula | Meaning |
|---|---:|---|
| Potential matrix | \(R(i,j)=E_i[N_j]\) | expected total visits to \(j\) |
| Hitting probability | \(F(i,j)=P_i(T_j<\infty)\) | probability of ever reaching \(j\) |
| Transient potential | \(S=(I-Q)^{-1}\) | expected visits among transient states |
| Absorption probabilities | \(G=SB\) | probabilities of entering recurrent classes |
| Invariant distribution | \(\pi=\pi P\) | distribution unchanged by one transition |
| Limiting distribution | \(\lim_n P^n(i,j)=\pi(j)\) | for irreducible aperiodic positive recurrent chains |
| Mean return time | \(\pi(j)=1/m(j)\) | long-run visit rate |
| Ergodic average | \(\frac1n\sum f(X_k)\to \pi f\) | time averages equal space averages |
| Periodic class | \(B_r\to B_{r+1}\) | cyclic motion prevents ordinary convergence |

# Checklist for solving Markov-chain limiting problems

1. **Classify states**: recurrent or transient.
2. **Find closed classes**: recurrent classes are closed communicating classes.
3. **If transient states exist**, extract \(Q\) and compute:
   \[
   S=(I-Q)^{-1}.
   \]
4. **Compute absorption probabilities**:
   \[
   G=SB.
   \]
5. **For each recurrent class**, check period.
6. **If aperiodic**, solve:
   \[
   \pi=\pi P,\qquad \sum_j\pi(j)=1.
   \]
7. **If periodic**, apply aperiodic theory to \(P^\delta\) on cyclic classes.
8. **For long-run averages**, use:
   \[
   \frac1n\sum_{k=0}^{n-1} f(X_k)\to \sum_j \pi(j)f(j).
   \]